# 2. Data Preprocessing

## Purpose
Transform raw data into a clean, numeric feature matrix ready for modelling.
Fits the preprocessing pipeline on training data only to prevent data leakage,
then applies it to both train and test sets.

## Inputs
- `data/raw/` — raw dataset (CSV)
- `config.yaml` — target column, columns to drop, split parameters, pipeline settings

## Outputs
- `data/processed/X_train.csv` — preprocessed training features
- `data/processed/X_test.csv` — preprocessed test features
- `data/processed/y_train.csv` — training labels
- `data/processed/y_test.csv` — test labels
- `artifacts/preprocessor.pkl` — fitted preprocessing pipeline

## Decisions for the user
- Set `target` in `config.yaml` — the column you are predicting
- Set `drop_columns` in `config.yaml` — ID columns or any columns to exclude
- Set `split.test_size` in `config.yaml` — typically 0.2 for most projects
- Set `split.stratify` in `config.yaml` — true for classification, false for regression
- Set `pipeline.scaler` in `config.yaml` — standard, minmax or robust
- Review the sparsity output — if density is very low consider dropping
  high missingness columns via `drop_columns`

## Important note
The test set is held out here and not used again until notebook 7.
All modelling decisions are made using cross validation scores only.

In [1]:
#Import required libraries and project API functions

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

from src.data.load_data import load_raw_data
from src.config.settings import load_config
from src.config.paths import ARTIFACTS_DIR, PROCESSED_DATA_DIR
from src.features.pipeline import FeatureEngineeringPipeline
from src.features.type_detection import detect_feature_types


## 1. Load config

In [2]:
config = load_config()

TARGET = config["target"]
DROP_COLS = config["drop_columns"]
SPLIT_CONFIG = config["split"]
PIPELINE_CONFIG = config["pipeline"]

print(f"Target:        {TARGET}")
print(f"Dropping:      {DROP_COLS}")
print(f"Test size:     {SPLIT_CONFIG['test_size']}")
print(f"Stratify:      {SPLIT_CONFIG['stratify']}")

Target:        Survived
Dropping:      ['PassengerId']
Test size:     0.2
Stratify:      True


## 2. Load data

In [3]:
df = load_raw_data() 
df.head()   

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Split features and target

In [4]:
X = df.drop(columns=[TARGET] + DROP_COLS)
y = df[TARGET]


## 4. Train/test split
The test set is held out completely until final evaluation.
Stratification preserves class balance between train and test sets.

In [5]:
stratify_col = y if SPLIT_CONFIG["stratify"] else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=SPLIT_CONFIG["test_size"],
    random_state=SPLIT_CONFIG["random_state"],
    stratify=stratify_col
)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print(f"\nTarget distribution in train:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTarget distribution in test:\n{y_test.value_counts(normalize=True).round(3)}")
     

Train size: (712, 10)
Test size:  (179, 10)

Target distribution in train:
Survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Target distribution in test:
Survived
0    0.615
1    0.385
Name: proportion, dtype: float64


## 5. Feature type detection
Automatically detects numeric, binary, categorical, high cardinality
and datetime features. Review the output before fitting the pipeline.

In [6]:
feature_types = detect_feature_types(X_train)
feature_types


C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project

{'numeric': ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'],
 'binary': [],
 'categorical': ['Sex', 'Embarked'],
 'high_cardinality': ['Name', 'Ticket', 'Cabin'],
 'datetime': []}

## 6. Configure and fit pipeline
The pipeline is fitted on training data only. It will be applied to
both train and test sets in the next step.

In [7]:
preprocessor = FeatureEngineeringPipeline(config=PIPELINE_CONFIG)
preprocessor.fit(X_train, target=y_train)


C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project

## 7. Transform train and test
The same fitted pipeline is applied to both sets ensuring identical
transformations without any information leakage from the test set.

In [8]:
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"X_train_transformed shape: {X_train_transformed.shape}")
print(f"X_test_transformed shape:  {X_test_transformed.shape}")

X_train_transformed shape: (712, 106)
X_test_transformed shape:  (179, 106)


C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}__hash_{j}"] = hashed[:, j]
C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}__hash_{j}"] = hashed[:, j]
C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has p

## 8. Check transformed features
Verify the transformed data has no NaN values, no infinite values
and reasonable sparsity before saving.

In [9]:
X_train_transformed.head()

,Pclass,Age,SibSp,Parch,Fare,Sex__male,Sex__female,Embarked__S,Embarked__C,Embarked__Q,...,Cabin__hash_22,Cabin__hash_23,Cabin__hash_24,Cabin__hash_25,Cabin__hash_26,Cabin__hash_27,Cabin__hash_28,Cabin__hash_29,Cabin__hash_30,Cabin__hash_31
692,0.828985,-0.090277,-0.464758,-0.465856,0.513451,1,0,1,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
481,-0.370684,-0.090277,-0.464758,-0.465856,-0.662098,1,0,1,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
527,-1.570353,-0.090277,-0.464758,-0.465856,3.952620,1,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
855,0.828985,-0.815155,-0.464758,0.727271,-0.467546,0,1,1,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
801,-0.370684,0.082312,0.477999,0.727271,-0.115895,0,1,1,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
#Check for NA's

train_nas = X_train_transformed.isna().sum().sum()
test_nas = X_test_transformed.isna().sum().sum()
print(f"NAs in train: {train_nas}")
print(f"NAs in test:  {test_nas}")  

NAs in train: 0
NAs in test:  0


In [11]:
#Check for infinite values

train_inf = np.isinf(X_train_transformed.to_numpy()).sum()
test_inf = np.isinf(X_test_transformed.to_numpy()).sum()
print(f"Infinite values in train: {train_inf}")
print(f"Infinite values in test:  {test_inf}")

Infinite values in train: 0
Infinite values in test:  0


In [12]:
#Check sparsity

density = (X_train_transformed != 0).sum().sum() / X_train_transformed.size
print(f"Train density: {density:.3f}")

# If density is below 0.1, consider:
# - Dropping high missingness columns via drop_columns in config.yaml
# - Reducing hashing_dim in config.yaml for high cardinality features
# - Extracting meaningful features before preprocessing (feature engineering notebook)


Train density: 0.094


## 9. Save processed data and preprocessor
Processed data is saved to `data/processed/` for use by all downstream
notebooks. The fitted preprocessor is saved to `artifacts/` so it can
be applied consistently to new data at prediction time.

In [13]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

X_train_transformed.to_csv(PROCESSED_DATA_DIR / "X_train.csv", index=False)
X_test_transformed.to_csv(PROCESSED_DATA_DIR / "X_test.csv", index=False)
y_train.to_csv(PROCESSED_DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DATA_DIR / "y_test.csv", index=False)

print(f"Processed data saved to {PROCESSED_DATA_DIR}")

# %%
# 13. Save Preprocessor
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, ARTIFACTS_DIR / "preprocessor.pkl")
print(f"Preprocessor saved to {ARTIFACTS_DIR / 'preprocessor.pkl'}")

Processed data saved to C:\ML\ML Workflow\Projects\Project Name Template\data\processed
Preprocessor saved to C:\ML\ML Workflow\Projects\Project Name Template\artifacts\preprocessor.pkl
